In [ ]:
import pandas as pd

df = pd.read_csv("dataset/creditcard_2023.csv")  # check file
# V1-V28: Anonymized features representing various transaction attributes (e.g., time, location, etc.)
df.head()

,id,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0,-0.260648,-0.469648,2.496266,-0.083724,0.129681,0.732898,0.519014,-0.130006,0.727159,...,-0.110552,0.217606,-0.134794,0.165959,0.126280,-0.434824,-0.081230,-0.151045,17982.10,0
1,1,0.985100,-0.356045,0.558056,-0.429654,0.277140,0.428605,0.406466,-0.133118,0.347452,...,-0.194936,-0.605761,0.079469,-0.577395,0.190090,0.296503,-0.248052,-0.064512,6531.37,0
2,2,-0.260272,-0.949385,1.728538,-0.457986,0.074062,1.419481,0.743511,-0.095576,-0.261297,...,-0.005020,0.702906,0.945045,-1.154666,-0.605564,-0.312895,-0.300258,-0.244718,2513.54,0
3,3,-0.152152,-0.508959,1.746840,-1.090178,0.249486,1.143312,0.518269,-0.065130,-0.205698,...,-0.146927,-0.038212,-0.214048,-1.893131,1.003963,-0.515950,-0.165316,0.048424,5384.44,0
4,4,-0.206820,-0.165280,1.527053,-0.448293,0.106125,0.530549,0.658849,-0.212660,1.049921,...,-0.106984,0.729727,-0.161666,0.312561,-0.414116,1.071126,0.023712,0.419117,14278.97,0


In [4]:
print(len(df))

568630


In [2]:
# Define features (X) and target (y)
# Drop 'id' (identifier) and 'Class' (target) for features
feature_cols = [c for c in df.columns if c not in ['id', 'Class']]
X = df[feature_cols]
y = df['Class']

# Split into train (80%) and test (20%)
# stratify=y keeps the same fraud/non-fraud ratio in both sets (important for imbalanced data)
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(f"Train: {X_train.shape[0]} samples, Test: {X_test.shape[0]} samples")
print(f"Train fraud rate: {y_train.mean():.4f}, Test fraud rate: {y_test.mean():.4f}")

Train: 454904 samples, Test: 113726 samples
Train fraud rate: 0.5000, Test fraud rate: 0.5000


In [ ]:
# Train/test split is in the cell above (X_train, X_test, y_train, y_test)

In [3]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Logistic Regression

In [5]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    class_weight=None,   # 불균형일 경우 "balanced"로 설정할 것.
    max_iter=1000
)

# 이 데이터(X_train)와 정답(y_train)을 보고, fraud를 가장 잘 구분하도록 내부 파라미터를 찾아라
model.fit(X_train_scaled, y_train)

# 불균형 데이터란?
# 예를 들어: 정상: 99%, fraud: 1%

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


### 불균형 데이터에선 왜 class_weight="balanced"?

불균형이면 모델은 이렇게 생각함: "99%는 정상인데 그냥 전부 정상이라 해도 되겠네?"

그래서 fraud를 무시할 수 있음.
class_weight="balanced"는 소수 클래스(fraud)에 더 큰 벌점 부여

즉, fraud를 틀리면 더 크게 혼나게 만드는 셈.

### 불균형 데이터에선 왜 ROC보다 PR-AUC?

ROC-AUC는 "전체적으로 얼마나 잘 구분하나?"를 본다.

하지만 불균형에서는 문제가 생김:

모델이 fraud를 거의 못 잡아도 정상을 잘 맞추면 ROC가 높게 나올 수 있음.

PR-AUC는 뭐 보냐면 -> "fraud를 얼마나 정확하게 잡고 있는가?"

즉, positive class 중심 지표라서 불균형에서 훨씬 민감하고 현실적

In [6]:
# 확률 예측
y_prob = model.predict_proba(X_test_scaled)[:, 1]

In [7]:
# 평가
from sklearn.metrics import roc_auc_score, average_precision_score

roc = roc_auc_score(y_test, y_prob)
pr = average_precision_score(y_test, y_prob)

print("ROC-AUC:", roc)
print("PR-AUC:", pr)

ROC-AUC: 0.9934995543387278
PR-AUC: 0.9944248997492257


In [9]:
# train 내부에서 cross-validation
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score

# 5-fold cross validation (총 5번 학습하고 평균을 낸다.)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", model)
])

scores = cross_val_score(
    pipe,
    X_train,   # 원본 (스케일링 안 된 것)
    y_train,
    cv=cv,
    scoring="average_precision"
)

print(scores.mean(), scores.std())

0.9946253287281651 0.00011205577641028385
